## [0] Imports, Data, and Functions

### [0.0] Imports

In [1]:
import torch
torch.set_default_dtype(torch.float64)
for i in range(torch.cuda.device_count()):
    print(f'Device: {torch.cuda.get_device_properties(i).name}')
if torch.cuda.is_available():
    i = 0
    torch.cuda.set_device(i)
    device = f'cuda:{i}'
    torch.set_default_device(f'cuda:{i}')
    print(f"Cuda is available. Setting default device to: {torch.cuda.get_device_properties(i).name}")
else:
    print('Cuda is not available. Setting default device to: CPU')
    device = 'cpu'

import kan
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

from kan_wrapper import KanModel, KanOptimizer, get_evaluator #type:ignore
from ax.service.ax_client import ObjectiveProperties

Device: NVIDIA GeForce RTX 4060 Laptop GPU
Cuda is available. Setting default device to: NVIDIA GeForce RTX 4060 Laptop GPU


### [0.1] CHF Data

In [2]:
# --- Loading Data
CHF = pd.read_csv("CHF_example/NE795_SciML_HW1_CHF_dataset_public.csv")
X_features = ["Tube Diameter", "Heated Length", "Pressure", "Mass Flux", "Outlet Quality", "Inlet Subcooling", "Inlet Temperature"]
Y_features = ["CHF"]
X_units = CHF.loc[0,X_features] #type:ignore
Y_units = CHF.loc[0,Y_features] #type:ignore
X_SCALER = StandardScaler()
Y_SCALER = StandardScaler()
X = CHF.loc[1:,X_features].astype(float).values
Y = CHF.loc[1:,Y_features].astype(float).values



In [3]:
# --- Test Train Split
X_TRAIN, X_TESTS, Y_TRAIN, Y_TESTS = train_test_split(X, Y, test_size=0.25)
X_TRAIN_SCALED = torch.tensor(X_SCALER.fit_transform(X_TRAIN)).to(device)
X_TESTS_SCALED = torch.tensor(X_SCALER.transform(X_TESTS)).to(device)
Y_TRAIN_SCALED = torch.tensor(Y_SCALER.fit_transform(Y_TRAIN)).to(device)
Y_TESTS_SCALED = torch.tensor(Y_SCALER.transform(Y_TESTS)).to(device)
DATASET = {'train_input':X_TRAIN_SCALED, 'test_input':X_TESTS_SCALED, 'train_label':Y_TRAIN_SCALED, 'test_label':Y_TESTS_SCALED}

In [4]:
print(f"X Features Shape:\t({X.shape[0]} Samples) x ({X.shape[1]} Features)\t\tSplit Into: ({X_TRAIN.shape[0]}/{X_TESTS.shape[0]}) Train/Test Samples")
print(f"Y Features Shape:\t({Y.shape[0]} Samples) x ({Y.shape[1]} Features)\t\tSplit Into: ({Y_TRAIN.shape[0]}/{Y_TESTS.shape[0]}) Train/Test Samples")

X Features Shape:	(24579 Samples) x (7 Features)		Split Into: (18434/6145) Train/Test Samples
Y Features Shape:	(24579 Samples) x (1 Features)		Split Into: (18434/6145) Train/Test Samples


### [0.2] FP Data

In [6]:
# --- Loading Data
# fp_in = pd.read_csv("../applications_datasets/fp_inp.csv")
# fp_out = pd.read_csv("../applications_datasets/fp_out.csv")

# X_features = ['fuel_dens','porosity','clad_thick','pellet_OD','pellet_h','gap_thick','inlet_T','enrich','rough_fuel','rough_clad','ax_pow','clad_T','pressure']
# Y_features = ['fis_gas_produced','max_fuel_centerline_temp','max_fuel_surface_temp','radial_clad_dia']
# X_units = fp_in.loc[0,X_features] #type:ignore
# Y_units = fp_out.loc[0,Y_features] #type:ignore
# X_SCALER = StandardScaler()
# Y_SCALER = StandardScaler()
# X = fp_in.loc[1:,X_features].astype(float).values
# Y = fp_out.loc[1:,Y_features].astype(float).values

In [7]:
# --- Test Train Split
# X_TRAIN, X_TESTS, Y_TRAIN, Y_TESTS = train_test_split(X, Y, test_size=0.25)
# X_TRAIN_SCALED = torch.tensor(X_SCALER.fit_transform(X_TRAIN), device='cuda:0')
# X_TESTS_SCALED = torch.tensor(X_SCALER.transform(X_TESTS), device='cuda:0')
# Y_TRAIN_SCALED = torch.tensor(Y_SCALER.fit_transform(Y_TRAIN), device='cuda:0')
# Y_TESTS_SCALED = torch.tensor(Y_SCALER.transform(Y_TESTS), device='cuda:0')
# DATASET = {'train_input':X_TRAIN_SCALED, 'test_input':X_TESTS_SCALED, 'train_label':Y_TRAIN_SCALED, 'test_label':Y_TESTS_SCALED}

In [8]:
# print(f"X Features Shape:\t({X.shape[0]} Samples) x ({X.shape[1]} Features)\t\tSplit Into: ({X_TRAIN.shape[0]}/{X_TESTS.shape[0]}) Train/Test Samples")
# print(f"Y Features Shape:\t({Y.shape[0]} Samples) x ({Y.shape[1]} Features)\t\tSplit Into: ({Y_TRAIN.shape[0]}/{Y_TESTS.shape[0]}) Train/Test Samples")

## [1] Optimizing

In [21]:
search_parameters = [{"name":"grid",
                      "type":"range",
                      "bounds":[3,12],
                      "value_type":"int",
                      "log_scale":False
                      },
                      {"name":"k",
                      "type":"range",
                      "bounds":[1,4],
                      "value_type":"int",
                      "log_scale":False
                      },
                      {"name":"lr",
                      "type":"range",
                      "bounds":[1e-3,1],
                      "value_type":"float",
                      "log_scale":False
                      },
                      {"name":"neurons",
                      "type":"range",
                      "bounds":[4,12],
                      "value_type":"int",
                      "log_scale":False
                      },
                      {"name":"layers",
                      "type":"range",
                      "bounds":[1,4],
                      "value_type":"int",
                      "log_scale":False
                      }]
objectives = {"MSE":ObjectiveProperties(minimize=True),}
              #"adjR2":ObjectiveProperties(minimize=False)}

In [22]:
evaluator = get_evaluator({"steps":100, "batch":100, "base_fun":'silu', "opt":'Adam'})

In [23]:
Optimizer = KanOptimizer(evaluator, search_parameters, DATASET, {'verbose_logging':False}, {'objectives':objectives, 'name':'Test'})

[INFO 02-26 08:51:13] ax.service.utils.instantiation: Created search space: SearchSpace(parameters=[RangeParameter(name='grid', parameter_type=INT, range=[3, 12]), RangeParameter(name='k', parameter_type=INT, range=[1, 4]), RangeParameter(name='lr', parameter_type=FLOAT, range=[0.001, 1.0]), RangeParameter(name='neurons', parameter_type=INT, range=[4, 12]), RangeParameter(name='layers', parameter_type=INT, range=[1, 4])], parameter_constraints=[]).
[INFO 02-26 08:51:13] ax.modelbridge.dispatch_utils: Using Models.BOTORCH_MODULAR since there is at least one ordered parameter and there are no unordered categorical parameters.
[INFO 02-26 08:51:13] ax.modelbridge.dispatch_utils: Calculating the number of remaining initialization trials based on num_initialization_trials=None max_initialization_trials=None num_tunable_parameters=5 num_trials=None use_batch_trials=False
[INFO 02-26 08:51:13] ax.modelbridge.dispatch_utils: calculated num_initialization_trials=10
[INFO 02-26 08:51:13] ax.mode

In [24]:
Optimizer.manual_step(parameters={"grid":3,
                                  "k":2,
                                  "lr":0.1,
                                  "neurons":5,
                                  "layers":1,
                                  },
                      index=0, eval_args={"device_from":"cuda","device_to":"cuda","outcomes":["MSE"]})

[INFO 02-26 08:51:18] ax.core.experiment: Attached custom parameterizations [{'grid': 3, 'k': 2, 'lr': 0.1, 'neurons': 5, 'layers': 1}] as trial 0.


checkpoint directory created: ./model
saving model version 0.0


| train_loss: 2.22e-01 | test_loss: 2.28e-01 | reg: 2.28e+01 | : 100%|█| 100/100 [00:01<00:00, 70.19

saving model version 0.1
R2: 0.9525394734408148	MAE: 0.14886417220271822	MSE: 0.046644176549184374	AdjR2 0.9524853388985443	RMSE: 0.21597262916671728	MaxAE: 699.1590841270167


In [26]:
Optimizer.fit(20, eval_args={"device_from":"cuda","device_to":"cuda","outcomes":["MSE"]})

checkpoint directory created: ./model
saving model version 0.0


| train_loss: 2.73e-01 | test_loss: 3.81e-01 | reg: 1.18e+02 | : 100%|█| 100/100 [00:01<00:00, 89.54


saving model version 0.1
R2: 0.8932905551807004	MAE: 0.22893612454049328	MSE: 0.10487397727056125	AdjR2 0.8931688399918891	RMSE: 0.32384251924440255	MaxAE: 1644.462906119442
checkpoint directory created: ./model
saving model version 0.0


| train_loss: 2.18e-01 | test_loss: 3.41e-01 | reg: 7.07e+01 | : 100%|█| 100/100 [00:01<00:00, 87.64


saving model version 0.1
R2: 0.910088635167028	MAE: 0.21637587333279096	MSE: 0.08836483450762776	AdjR2 0.9099860802454326	RMSE: 0.2972622318889969	MaxAE: 426.28289381034034
checkpoint directory created: ./model
saving model version 0.0


...

| train_loss: 2.57e-01 | test_loss: 1.94e-01 | reg: 2.22e+01 | : 100%|█| 100/100 [00:01<00:00, 86.49


saving model version 0.1
R2: 0.9641634097266517	MAE: 0.1279159827505011	MSE: 0.03522017905861893	AdjR2 0.9641225337071123	RMSE: 0.18767040005983612	MaxAE: 1317.389377997802
checkpoint directory created: ./model
saving model version 0.0


| train_loss: 2.42e-01 | test_loss: 2.35e-01 | reg: 3.58e+01 | : 100%|█| 100/100 [00:01<00:00, 77.92


saving model version 0.1
R2: 0.9518439560172475	MAE: 0.15832759936642501	MSE: 0.047327730648767594	AdjR2 0.9517890281521865	RMSE: 0.21754937519737352	MaxAE: 158.71901990142973


({'grid': 4, 'k': 2, 'lr': 0.1275918318888153, 'neurons': 4, 'layers': 2},
 ({'MSE': 0.037544111428892385}, {'MSE': {'MSE': 0.0003103037488572007}}))